In [2]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [3]:
event_log_name = "medium"
log_path = f"D:\\LTNcoder\\.out\\eventlogs\\{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

c:\Users\devas\anaconda3\envs\ltn\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [4]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [5]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

File medium-0.3-1.decl does not exist, running discovery...
Computing discovery ...
Total constraints discovered: 1763


In [6]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


File medium_5000_conformance_results.pkl does not exist, running conformance checking...
Conformance checking results saved to medium_5000_conformance_results.pkl


In [7]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


C:\Users\devas\AppData\Local\Temp\ipykernel_101924\1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [8]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print("Filtered Metrics DataFrame:")
display(filtered_metrics_df)

Filtered Metrics DataFrame:


,support,confidence
"Responded Existence[Activity N, Activity J] | |",0.0524,0.988679
"Responded Existence[Activity N, Activity B] | |",0.0524,0.988679
"Responded Existence[Activity H, Activity B] | |",0.0750,0.986842
"Responded Existence[Activity G, Activity B] | |",0.1542,0.985934
"Response[Activity N, Activity B] | |",0.0522,0.984906
...,...,...
"Precedence[Activity W, Activity V] | |",0.1884,0.733645
"Alternate Response[Activity U, Activity W] | |",0.1870,0.730469
"Response[Activity U, Activity W] | |",0.1870,0.730469
"Alternate Precedence[Activity W, Activity V] | |",0.1868,0.727414


In [16]:
raise KeyboardInterrupt("\nStopping execution after displaying filtered metrics DataFrame.\nChoose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.\nThen run the cells below again to see the results of the selected constraints.")

KeyboardInterrupt: 
Stopping execution after displaying filtered metrics DataFrame.
Choose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.
Then run the cells below again to see the results of the selected constraints.

# Constraints with low support and high confidence
1. Responded Existence[Activity H, Activity B] | |	0.075	0.9868421052631579
2. Response[Activity N, Activity B] | |	0.0522	0.9849056603773585
3. Responded Existence[Activity N, Activity J] | |	0.0524	0.9886792452830189

In [10]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    "Responded Existence[Activity H, Activity B] | |"
    ]

In [11]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[Activity H, Activity B] | |    375
dtype: int64
[10, 17, 21, 41, 59, 61, 85, 92, 95, 97, 127, 139, 148, 150, 164, 189, 195, 204, 220, 237, 254, 271, 289, 292, 293, 299, 307, 309, 323, 330, 358, 374, 376, 392, 394, 398, 409, 412, 413, 418, 493, 497, 498, 511, 514, 515, 537, 578, 585, 605, 654, 658, 665, 690, 696, 713, 725, 728, 732, 770, 773, 807, 822, 843, 861, 870, 873, 900, 903, 919, 923, 953, 986, 995, 1010, 1026, 1037, 1065, 1069, 1080, 1091, 1095, 1100, 1110, 1119, 1125, 1163, 1171, 1180, 1186, 1200, 1222, 1283, 1312, 1327, 1331, 1356, 1408, 1413, 1433, 1439, 1522, 1526, 1533, 1541, 1544, 1559, 1564, 1576, 1588, 1589, 1594, 1604, 1613, 1631, 1670, 1719, 1737, 1749, 1760, 1772, 1775, 1776, 1778, 1780, 1812, 1822, 1836, 1858, 1885, 1901, 1908, 1938, 1968, 1969, 1984, 1987, 1992, 1997, 2031, 2043, 2045, 2054, 2055, 2060, 2065, 2073, 2080, 2081, 2085, 2107, 2133, 2162, 2177, 2200, 2202, 2207, 2242, 2246, 2247, 2261, 2263, 2279, 2281, 2282, 2291, 2308, 2311, 2348, 2

In [12]:
print("END")

END


In [13]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)